<a href="https://colab.research.google.com/github/caramos84/QC_Video/blob/main/DecisionEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QC Video - Notebook 04
## Asset Knowledge Engine

Este notebook consolida todos los resultados obtenidos por los notebooks anteriores en un único modelo de conocimiento del asset.

Entradas

- output.zip
- visual_analysis.json
- frame_analysis/
- audio_analysis.json
- transcript.json
- metadata.json
- manifest.json

Salida

asset_knowledge.json

In [1]:
# Librerías

import os
import json
import shutil
import zipfile

from google.colab import files

## Cargar paquete de resultados

In [2]:
uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

Saving output_audio_analysis.zip to output_audio_analysis.zip
Saving output_visual_analysis.zip to output_visual_analysis.zip


In [10]:
OUTPUT="/content/output"

# Clear existing output directory if it exists
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
# Create it again to ensure it's there, as extractall might create it, but good to be explicit.
os.makedirs(OUTPUT, exist_ok=True)

# Iterate through all uploaded files and extract zips
for file_name in uploaded.keys():
    if file_name.endswith(".zip"):
        with zipfile.ZipFile(file_name,"r") as z:
            # Extract content to /content. Assuming the zip files contain a root folder 'output'
            # or directly the files like 'visual_analysis.json' that should go into /content/output.
            z.extractall("/content")

## Función para cargar JSON

In [4]:
def load_json(path):

    if os.path.exists(path):

        with open(path,"r",encoding="utf8") as f:

            return json.load(f)

    return None

## Cargar artefactos existentes

In [5]:
metadata = load_json(
    "/content/output/metadata.json"
)

manifest = load_json(
    "/content/output/manifest.json"
)

visual = load_json(
    "/content/output/visual_analysis.json"
)

audio = load_json(
    "/content/output/audio_analysis.json"
)

transcript = load_json(
    "/content/output/transcript.json"
)

## Leer análisis por frame

In [6]:
frame_folder="/content/output/frame_analysis"

frames=[]

if os.path.exists(frame_folder):

    for file in sorted(os.listdir(frame_folder)):

        frames.append(

            load_json(

                os.path.join(

                    frame_folder,

                    file

                )

            )

        )

## Construcción del Knowledge Model

In [7]:
asset_knowledge={

    "asset":{

        "metadata":metadata,

        "manifest":manifest

    },

    "visual":{

        "summary":visual,

        "frames":frames

    },

    "audio":{

        "summary":audio,

        "transcript":transcript

    }

}

## Guardar asset_knowledge.json

In [8]:
knowledge_path="/content/output/asset_knowledge.json"

with open(

    knowledge_path,

    "w",

    encoding="utf8"

) as f:

    json.dump(

        asset_knowledge,

        f,

        indent=4,

        ensure_ascii=False

    )

print("Knowledge generado.")

Knowledge generado.


## Resumen del Asset

In [11]:
summary={

    "archivo":

        metadata["filename"],

    "duracion":

        metadata["duration"],

    "resolucion":

        f"{metadata['width']}x{metadata['height']}",

    "frames":

        len(frames),

    "texto_detectado":

        visual["total_words"] if visual and "total_words" in visual else 0,

    "palabras_audio":

        audio["word_count"],

    "idioma":

        audio["language"]

}

summary

{'archivo': 'OFERTA 1 ESCOLAR 6.01.26_V3.mp4',
 'duracion': 10.01,
 'resolucion': '1920x1080',
 'frames': 0,
 'texto_detectado': 0,
 'palabras_audio': 32,
 'idioma': 'es'}

## Exportar paquete enriquecido

In [12]:
!zip -r asset_knowledge.zip output > /dev/null

files.download("asset_knowledge.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>